Weather Data Cleaning & Formatting
-----------------------------------------------------------------------------------------------------------------------------------
Weather data from the Veenkapmen station in the Binneveld near Wageningen has been exported from the Meteorology & Air Quality (MAQ) group from the WUR.

In this notebook the methodology used to convert the units of this raw data is represented, and is exported again in according format. This step is needed such that the meteorological data can be succesfully implemented in the Python Crop Simulation Environment (PCSE) and be read effectively by the ExcelWeatherDataProvider function.

In [17]:
# READ RAW WEATHER DATA FROM CSV FILES
# ________________________________________________________________________________________________________________________________
import pandas as pd 
import glob
from pathlib import Path
import requests
import openpyxl
from datetime import datetime

folder = Path("/Users/paulianoprescu/Projects/wofost_miscanthus/miscanthus_calibration/raw_data/weather")

dfs = []
for f in folder.glob("*.csv"):
    df = pd.read_csv(f, skiprows=[1], parse_dates=["Timestamp"], low_memory=False)
    df = df.set_index("Timestamp")
    df = df.apply(pd.to_numeric, errors="coerce")
    dfs.append(df)

data = dfs[0].join(dfs[1:], how="outer")
data.head()

,WS_1_2_1,SW_IN_1_1_1,e,TA_2_1_1,P_1_1_7
Timestamp,,,,,
2016-01-01 00:00:00,3.201,-1.681,734.366105,4.992,0.0
2016-01-01 00:01:00,3.630,-1.550,NaN,5.081,0.0
2016-01-01 00:02:00,3.405,-1.465,NaN,5.153,0.0
2016-01-01 00:03:00,3.315,-1.448,NaN,5.199,0.0
2016-01-01 00:04:00,3.063,-1.419,NaN,5.225,0.0


In [18]:
# UNIT CONVERSION
# ________________________________________________________________________________________________________________________________
# Irradiation variable required in kJ/m2/day
# Veenkampen station has measured incoming shortwave radiation in W m-2 each minute, using Kipp&Zonen CM11 upgraded. 
# Unit conversion required :

# 1 W = 1 J/s
# Convert W/m2 to J/m2 per minute: J/m2/min = W/m2 * 60 s
# Convert J/m2/min to kJ/m2/min: kJ/m2/min = J/m2/min * 0.001
data["IRRAD"] = data["SW_IN_1_1_1"] * 0.06 

# Group all observations by day and sum the irradiation values to get daily irradiation in kJ/m2/day
daily_data = data[["IRRAD"]].resample("D").sum()

# ________________________________________________________________________________________________________________________________
# Daily minimum & maximum temperature variables required in °C
# Veenkampen station has measured dry bulb temperature ventilated at 1.5 m in °C each minute using PT100.
daily_data["TMIN"] = data[["TA_2_1_1"]].resample("D").min()
daily_data["TMAX"] = data[["TA_2_1_1"]].resample("D").max()

# ________________________________________________________________________________________________________________________________
# Daily mean vapour pressure variable required in kPa
# Veenkampen station has measured ambient water vapor partial pressure in Pa each half hour using Licor Li-7500 / Campbell CSAT3.

# Convert Pa to kPa: kPa = Pa * 0.001
daily_data["VAP"] = data[["e"]].resample("D").mean() * 0.001

# ________________________________________________________________________________________________________________________________
# Daily mean windspeed at 2 m above the surface variable required in m/s
# Veenkampen station has measured the mean wind speed at 2m in m/s each minute using WindSonic 75.
daily_data["WIND"] = data[["WS_1_2_1"]].resample("D").mean()

# ________________________________________________________________________________________________________________________________
# Daily rainfall sum variable required in mm
# Veenkampen station has measured rainfall in mm each minute using a precipitation weighting gauge (Ott Pluvio S, no heating)
daily_data["RAIN"] = data[["P_1_1_7"]].resample("D").sum()

# ________________________________________________________________________________________________________________________________
# Add snowdepth variable
daily_data["SNOWDEPTH"] = -999

# ________________________________________________________________________________________________________________________________
# Classify negative irradiation values (likely erroneous) as NaN for AgERA5 replacement
daily_data["IRRAD"] = daily_data["IRRAD"].where(daily_data["IRRAD"] > 0) 

daily_data.tail()

,IRRAD,TMIN,TMAX,VAP,WIND,RAIN,SNOWDEPTH
Timestamp,,,,,,,
2026-03-11,4029.71994,4.445,11.599,1.167046,3.707676,4.85,-999
2026-03-12,12296.93940,3.130,12.685,1.004923,5.242832,0.06,-999
2026-03-13,3110.22282,4.724,10.398,1.017070,5.051580,6.21,-999
2026-03-14,4975.64190,-0.246,8.085,0.827735,1.517324,6.49,-999
2026-03-15,13668.29562,-1.522,11.009,0.920601,3.083594,2.55,-999


In [21]:
# Find Missing Values : Specific Variable & Date Range
# Import AgERA5 data for missing values; API = /api/v1/get_agera5 ; Wageningen Coordinates = 51.97°N, 5.67°E

# Note: This URL can only be accessed from the WUR network
base_url = "https://agera5.containers.wurnet.nl/"

# Wageningen coordinates
lat, lon = 51.97, 5.67

# API Readibility Check
try:
    test = requests.get(f"{base_url}api/v1/get_agera5?latitude={lat}&longitude={lon}&startdate=20200101&enddate=20200102", timeout=10)
    test.raise_for_status()
    print("API is accessible and responsive.")
except requests.exceptions.RequestException as e:
    print(f"API access error - are you on the WUR network? Error: {e}")

# AgERA5 variable name mapping to PCSE variables
agera5_map = {
    "IRRAD": "radiation",
    "TMIN": "temperature_min",
    "TMAX": "temperature_max",
    "VAP": "vapourpressure",
    "WIND": "windspeed",
    "RAIN": "precipitation"
}

# Identify the different data ranges that have missing values
mask_any = daily_data[list(agera5_map.keys())].isna().any(axis=1)
if mask_any.any():
    start = mask_any[mask_any].index.min().strftime("%Y-%m-%d")
    end = mask_any[mask_any].index.max().strftime("%Y-%m-%d")

    resp = requests.get(f"{base_url}api/v1/get_agera5?latitude={lat}&longitude={lon}&startdate={start}&enddate={end}")
    resp.raise_for_status()
    agera5 = pd.DataFrame(resp.json()["weather_variables"])
    agera5.index = pd.to_datetime(agera5["day"])

    # Insert AgERA5 data into the daily_data DataFrame for the missing date range
    for col, agera5_col in agera5_map.items():
        missing = daily_data[col].isna()
        if missing.any():
            # Only fill dates that AgERA5 actually has
            common = missing[missing].index.intersection(agera5.index)
            if len(common) > 0:
                fill_values = agera5.loc[common, agera5_col].astype(float)
                if col == "VAP":
                    fill_values = fill_values * 0.001  # Convert AgERA5 vapour pressure from hPa to kPa (PCSE format)
                if col == "IRRAD":
                    fill_values = fill_values / 1000  # J/m²/day -> kJ/m²/day (PCSE format) : ASSUMPTION
                daily_data.loc[common, col] = fill_values
                print(f"{col}: filled {len(common)} days from AgERA5")

            # Report any remaining gaps
            still_missing = daily_data[col].isna().sum()
            if still_missing > 0:
                print(f"  -> {still_missing} days still missing (beyond AgERA5 coverage)")

# Interpolation of missing & erroneous values in the daily data using time-based interpolation (are not captured by AgERA5)
daily_data["IRRAD"] = daily_data["IRRAD"].interpolate(method="time")
daily_data["TMIN"] = daily_data["TMIN"].interpolate(method="time")
daily_data["TMAX"] = daily_data["TMAX"].interpolate(method="time")
daily_data["VAP"] = daily_data["VAP"].interpolate(method="time")
daily_data["WIND"] = daily_data["WIND"].interpolate(method="time")
daily_data["RAIN"] = daily_data["RAIN"].interpolate(method="time")

API is accessible and responsive.
IRRAD: filled 11 days from AgERA5
TMIN: filled 2 days from AgERA5
TMAX: filled 2 days from AgERA5
VAP: filled 149 days from AgERA5
  -> 1 days still missing (beyond AgERA5 coverage)
WIND: filled 2 days from AgERA5


In [23]:
# Check AgERA5 radiation units
test = requests.get(
    f"{base_url}api/v1/get_agera5",
    params=dict(latitude=51.97, longitude=5.67, startdate="20200601", enddate="20200603"),
    timeout=10,
)
test_data = pd.DataFrame(test.json()["weather_variables"])
print("AgERA5 radiation values (documented as kJ/m²/day):")
print(test_data[["day", "radiation"]])
print("\nThese values are ~1000x too high for kJ/m²/day → API returns J/m²/day")

AgERA5 radiation values (documented as kJ/m²/day):
                   day   radiation
0  2020-06-01T00:00:00  27175384.0
1  2020-06-02T00:00:00  25801146.0
2  2020-06-03T00:00:00  22420496.0

These values are ~1000x too high for kJ/m²/day → API returns J/m²/day


In [22]:
# WEATHER DATA EXPORT
# ________________________________________________________________________________________________________________________________
# Open excel file and select the PCSE ExcelWeatherDataProvider readable weather input sheet for WOFOST
wb = openpyxl.load_workbook("/Users/paulianoprescu/Projects/wofost_miscanthus/miscanthus_calibration/input_params/weather/veenkampen_weather.xlsx")
ws = wb["observed_daily_weather"]

# Write the transformed daily weather data into the selected sheet, starting at row 13, column 1
for i, (date,row) in enumerate(daily_data.iterrows()):
    r = i + 13
    ws.cell(row=r, column=1, value=date.to_pydatetime())
    ws.cell(row=r, column=2, value=row["IRRAD"])
    ws.cell(row=r, column=3, value=row["TMIN"])
    ws.cell(row=r, column=4, value=row["TMAX"])
    ws.cell(row=r, column=5, value=row["VAP"])
    ws.cell(row=r, column=6, value=row["WIND"])
    ws.cell(row=r, column=7, value=row["RAIN"])
    ws.cell(row=r, column=8, value=row["SNOWDEPTH"])

# Save the modified excel file
wb.save("/Users/paulianoprescu/Projects/wofost_miscanthus/miscanthus_calibration/input_params/weather/veenkampen_weather.xlsx")